<img src="https://drive.google.com/uc?export=view&id=1wYSMgJtARFdvTt5g7E20mE4NmwUFUuog" width="200">

[![Gen AI Experiments](https://img.shields.io/badge/Gen%20AI%20Experiments-GenAI%20Bootcamp-blue?style=for-the-badge&logo=artificial-intelligence)](https://github.com/buildfastwithai/gen-ai-experiments)
[![GitHub Stars](https://img.shields.io/github/stars/buildfastwithai/gen-ai-experiments?style=for-the-badge&logo=github&color=gold)](http://github.com/buildfastwithai/gen-ai-experiments)

# 🛠️ Day 4: Function Calling — How AI Uses Tools
### 30-Day #LearnAI Series | BuildFastWithAI

**Learning Objectives:**
- Understand what function calling is and why it matters
- See the before/after with a real weather app example
- Learn the 3-step loop: Ask → Detect → Call → Respond
- Build your own tool-enabled AI assistant

👉 [Master GenAI Fundamentals](https://www.buildfastwithai.com/genai-course)

---
## 🤔 What is Function Calling?

By default, AI models only know what they were trained on. They **cannot**:
- Check today's weather 🌦️
- Look up stock prices 📈
- Search your database 🗃️
- Book a meeting 📅

**Function Calling** lets you give AI a "menu of tools" it can use.

When the AI recognizes it needs external data, it:
1. Picks the right tool from your menu
2. Tells you exactly what arguments to use
3. Waits for you to run it and return the result
4. Uses that result to write its final answer

> 💡 **Key insight:** The AI never runs your code. It only *requests* the call. You are always in control.

---
## ⚙️ Setup

In [1]:
# Install dependencies
!pip install openai -q

In [2]:
import json
from openai import OpenAI
import os

# If running on Google Colab:
from google.colab import userdata
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', 'your-api-key-here'))
print('✅ Setup complete!')

✅ Setup complete!


---
## ❌ BEFORE: AI Without Function Calling

Let's first see what happens when we ask about the weather **without** any tools.

In [3]:
# BEFORE: Plain AI call - no tools, no real-time data
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role': 'user', 'content': "What's the weather like in Mumbai right now?"}]
)
print('🤖 AI Response (without tools):')
print(response.choices[0].message.content)

🤖 AI Response (without tools):
I'm unable to provide real-time updates or current weather information. To find out the current weather in Mumbai, I recommend checking a reliable weather website or using a weather app.


**Notice:** The AI says it does not have real-time data. It is stuck in its training bubble.

Now let's fix that with function calling. 👇

---
## ✅ AFTER: AI With Function Calling

### Step 1 — Define Your Tools

You give the AI a "menu" — a list of functions it is allowed to request.

In [4]:
# Step 1: Define the tools (what the AI is ALLOWED to call)
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_current_weather',
            'description': 'Get the current weather conditions for a given city',
            'parameters': {
                'type': 'object',
                'properties': {
                    'location': {
                        'type': 'string',
                        'description': 'City name, e.g. Mumbai, New York, Tokyo'
                    },
                    'unit': {
                        'type': 'string',
                        'enum': ['celsius', 'fahrenheit'],
                        'description': 'Temperature unit'
                    }
                },
                'required': ['location']
            }
        }
    }
]

print('🛠️ Tool defined: get_current_weather')
print(f"Parameters: {list(tools[0]['function']['parameters']['properties'].keys())}")

🛠️ Tool defined: get_current_weather
Parameters: ['location', 'unit']


### Step 2 — Write the Actual Function

This is **your** code. In production, replace mock data with a real API (e.g. OpenWeatherMap).

In [5]:
def get_current_weather(location: str, unit: str = 'celsius') -> dict:
    """Mock weather function - replace with real API call in production"""
    mock_data = {
        'mumbai':   {'temp': 32, 'condition': 'Sunny',         'humidity': '78%', 'wind': '12 km/h'},
        'new york': {'temp': 8,  'condition': 'Cloudy',        'humidity': '65%', 'wind': '20 km/h'},
        'tokyo':    {'temp': 15, 'condition': 'Partly Cloudy', 'humidity': '55%', 'wind': '8 km/h'},
        'london':   {'temp': 10, 'condition': 'Rainy',         'humidity': '85%', 'wind': '25 km/h'},
        'dubai':    {'temp': 38, 'condition': 'Clear',         'humidity': '40%', 'wind': '5 km/h'},
    }
    w = mock_data.get(location.lower(), {'temp': 22, 'condition': 'Clear', 'humidity': '60%', 'wind': '10 km/h'})
    temp = f"{round(w['temp'] * 9/5 + 32)}°F" if unit == 'fahrenheit' else f"{w['temp']}°C"
    return {'location': location.title(), 'temperature': temp, 'condition': w['condition'], 'humidity': w['humidity'], 'wind_speed': w['wind']}

# Test it directly
print(get_current_weather('Mumbai'))

{'location': 'Mumbai', 'temperature': '32°C', 'condition': 'Sunny', 'humidity': '78%', 'wind_speed': '12 km/h'}


### Step 3 — The Complete Function Calling Loop

**3 rounds:**
1. **Round 1:** You ask → AI recognises it needs a tool
2. **Round 2:** You run the tool → feed the result back
3. **Round 3:** AI writes the final natural-language answer

In [6]:
def ask_with_tools(user_question: str) -> str:
    """Complete function calling loop: Ask -> Detect -> Call -> Respond"""
    print(f"\n" + '='*55)
    print(f'👤 User: {user_question}')
    print('='*55)

    # Round 1: Ask AI (with tools attached)
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': user_question}],
        tools=tools,
        tool_choice='auto'
    )
    message = response.choices[0].message

    if not message.tool_calls:
        print('💬 AI answered directly (no tool needed)')
        print(message.content)
        return message.content

    # Round 2: Run the tool AI requested
    messages = [{'role': 'user', 'content': user_question}, message]
    for tc in message.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f'\n🔧 AI requested: {tc.function.name}({args})')
        result = get_current_weather(**args) if tc.function.name == 'get_current_weather' else {'error': 'unknown'}
        print(f'📦 Result: {result}')
        messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': json.dumps(result)})

    # Round 3: AI reads result and answers
    final = client.chat.completions.create(model='gpt-4o', messages=messages)
    answer = final.choices[0].message.content
    print(f'\n🤖 AI Answer:\n{answer}')
    return answer


ask_with_tools("What's the weather like in Mumbai right now?")


👤 User: What's the weather like in Mumbai right now?

🔧 AI requested: get_current_weather({'location': 'Mumbai', 'unit': 'celsius'})
📦 Result: {'location': 'Mumbai', 'temperature': '32°C', 'condition': 'Sunny', 'humidity': '78%', 'wind_speed': '12 km/h'}

🤖 AI Answer:
The current weather in Mumbai is sunny with a temperature of 32°C. The humidity level is at 78% and there is a wind speed of 12 km/h.


'The current weather in Mumbai is sunny with a temperature of 32°C. The humidity level is at 78% and there is a wind speed of 12 km/h.'

---
## 🧪 Try More Questions!

In [7]:
questions = [
    'Is it a good day to carry an umbrella in Tokyo?',
    'Compare the weather in New York and Dubai',
    'What is 2 + 2?',        # Will NOT use the tool (AI answers directly)
    'Should I wear a jacket in London today?',
]

for q in questions:
    ask_with_tools(q)


👤 User: Is it a good day to carry an umbrella in Tokyo?

🔧 AI requested: get_current_weather({'location': 'Tokyo'})
📦 Result: {'location': 'Tokyo', 'temperature': '15°C', 'condition': 'Partly Cloudy', 'humidity': '55%', 'wind_speed': '8 km/h'}

🤖 AI Answer:
It is currently partly cloudy in Tokyo with a moderate temperature of 15°C, humidity at 55%, and wind speed at 8 km/h. It may not be necessary to carry an umbrella today, as there's no indication of rain in the current conditions. However, it's always a good idea to check a detailed forecast if you'll be out for a while or if weather conditions change rapidly.

👤 User: Compare the weather in New York and Dubai

🔧 AI requested: get_current_weather({'location': 'New York', 'unit': 'celsius'})
📦 Result: {'location': 'New York', 'temperature': '8°C', 'condition': 'Cloudy', 'humidity': '65%', 'wind_speed': '20 km/h'}

🔧 AI requested: get_current_weather({'location': 'Dubai', 'unit': 'celsius'})
📦 Result: {'location': 'Dubai', 'temperatu

---
## 🔑 Key Concepts Recap

| Concept | Meaning |
|---------|--------|
| `tools` | Menu of functions AI can request |
| `tool_choice='auto'` | AI decides when to use a tool |
| `tool_calls` in response | AI is requesting a function call |
| `role: 'tool'` | How you feed the result back to AI |
| AI runs your code? | **Never.** AI only requests it. |

---

### 🌍 Real-World Use Cases

```
🛒 E-commerce  -> search_products('red sneakers size 10')
📅 Scheduling  -> book_meeting(date, time, attendees)
💰 Finance     -> get_stock_price('AAPL')
🗃️ Support    -> lookup_order(order_id)
🔍 Research   -> search_web(query)
```

The AI handles the **language**. You handle the **data**.

---
## 🏋️ Challenge: Add a Second Tool!

Try adding a `get_weather_forecast` tool that returns a 3-day forecast.  
Hint: add it to the `tools` list and handle it in the `if tc.function.name ==` block.

In [ ]:
# Your code here!
def get_weather_forecast(location: str, days: int = 3) -> dict:
    # TODO: return a forecast dict, e.g. {'Day 1': 'Sunny', 'Day 2': 'Cloudy', ...}
    pass